# Data Analysis using __PySpark__  
*Fun with the __MovieLens__ dataset*  

**Harder Problems, Moar Fun**

<font color='green'>__Support for Google Colab__  </font>

open this notebook in Colab using the following button:  
  
<a href="https://colab.research.google.com/github/shauryashaurya/learn-data-munging/blob/main/03-Spark/003.01-Harder-Problems-in-PySpark(Questions-only).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>  

<font color='green'>uncomment and execute the cell below to setup and run this Spark notebook on Google Colab.</font>

# Setup, Loading Data, yada yada... 

we've done this before...

## update Jan 2025 
It has been very frustrating to run spark 3.5.x with Python 3.12  
While the official documentation says it works, the python workers for pySpark have been crashing randomly.   
Not cool!  

So for the time being we'll stick with Spark 3.5.4 (ditto for pySpark - v3.5.4) with Python 3.11.xSo for the time being we'll stick with Spark 3.5.4 (ditto for pySpark - v3.5.4) with Python 3.11.x
See [this](https://www.reddit.com/r/dataengineering/comments/1dupogi/wasted_45_hours_to_install_pyspark_locally_pain/) reddit post that shares my frustration.  

In [1]:
# # SETUP FOR COLAB: select all the lines below and uncomment (CTRL+/ on windows)

# # grab spark
# # as of Jan 2025, the *working* version is 3.5.4, get the link from Apache Spark's website
# ! wget -q https://dlcdn.apache.org/spark/spark-3.5.4/spark-3.5.4-bin-hadoop3.tgz
# # unzip spark
# !tar xf spark-3.5.4-bin-hadoop3.tgz
# # install findspark package
# !pip install -q findspark
# # Let's download and unzip the MovieLens 25M Dataset as well.
# ! mkdir ./../data
# ! wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip
# ! unzip ./ml-25m.zip -d ./../data/

# # got to provide JAVA_HOME and SPARK_HOME vairables
# import os
# # IMPORTANT - check the version of java, use 11 or 17, code not tested on 21 yet
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
# # IMPORTANT - UPDATE THE SPARK_HOME PATH BASED ON THE PACKAGE YOU DOWNLOAD
# os.environ["SPARK_HOME"] = "/content/spark-3.5.4-bin-hadoop3"
# ! echo "DONE"

**Citation**:  
*F. Maxwell Harper and Joseph A. Konstan.* 2015.  
The MovieLens Datasets: History and Context.  
ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>  

## Setup the Spark Cluster

In [2]:
# Step 1: initialize findspark
import findspark

findspark.init()

In [3]:
# Step 2: import pyspark
import pyspark
from pyspark.sql import SparkSession

pyspark.__version__

'3.5.4'

In [4]:
# Step 3: Create a spark session

# using local[*] to use as many logical cores as available, use 1 when in doubt
# 'local[1]' indicates spark on 1 core on the local machine or specify the number of cores needed
# use .config("spark.some.config.option", "some-value") for additional configuration

spark = (
    SparkSession.builder.master("local[1]")
    .appName("Analyzing Movielens Data")
    .getOrCreate()
)

In [5]:
# spark

## Load data, carefull with the schema

### Schema Spec

Here's the list of files (as of Aug 2022) that you get when you unzip the dataset:
1. **movies**.csv - list of movies with at least one rating.  
    Header: ```movieId,title,genres```  
1. **links**.csv - IDs to generate links to the movie listing on imdb.com and themoviedb.org  
    Header: ```movieId,imdbId,tmdbId```  
1. **ratings**.csv - Each line of this file after the header row represents one rating of one movie by one user.  
    Header: ```userId,movieId,rating,timestamp```  
1. **tags**.csv - Each line of this file after the header row represents one tag applied to one movie by one user.  
    Header: ```userId,movieId,tag,timestamp```  
1. Tag Genome: The tag genome contains tag relevance scores for movies. See [this](http://files.grouplens.org/papers/tag_genome.pdf)  
	1. **genome-tags**.csv - A list of tags  
    Header: ```tagId,tag```  
	1. **genome-scores**.csv - Each movie in the genome has a relevance score value for every tag in the genome  
    Header: ```movieId,tagId,relevance```  
1. README.txt - Check out the README.txt for more details about the files.  

### Data encoding details

From the Readme file, we have the following observations about the data:
1. Each file is a CSV with a single header row
1. Separator char is ```,```
1. Escape char is ```"```
1. Encoding is UTF-8

Let's set these options when reading the CSV files.

### Specify the schema for Spark  
  
Avoid ```inferSchema``` as much as possible, just cleaner

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.rdd import RDD

In [7]:
#
schema_movies = StructType(
    [
        StructField("movieId", StringType(), False),
        StructField("title", StringType(), False),
        StructField("genres", StringType(), True),
    ]
)

In [8]:
#
schema_links = StructType(
    [
        StructField("movieId", StringType(), False),
        StructField("imdbId", StringType(), True),
        StructField("tmdbId", StringType(), True),
    ]
)

In [9]:
#
schema_ratings = StructType(
    [
        StructField("userId", StringType(), False),
        StructField("movieId", StringType(), False),
        StructField("rating", FloatType(), True),
        StructField("timestamp", StringType(), True),
    ]
)

In [10]:
#
schema_tags = StructType(
    [
        StructField("userId", StringType(), False),
        StructField("movieId", StringType(), False),
        StructField("tag", StringType(), True),
        StructField("timestamp", StringType(), True),
    ]
)

In [11]:
#
schema_genome_tags = StructType(
    [
		StructField("tagId", StringType(), False), 
		StructField("tag", StringType(), False)
	]
)

In [12]:
#
# using arbitrary precision signed decimals (java.math.BigDecimal) for relevance scores
schema_genome_scores = StructType(
    [
        StructField("movieId", StringType(), False),
        StructField("tagId", StringType(), False),
        StructField("relevance", DecimalType(), False),
    ]
)

### Specify the location of your data  

Change this folder if you are saving the data at the different place

In [13]:
datalocation = "../data/ml-25m/"

In [14]:
# specify file names
file_path_movies = datalocation + "movies.csv"
file_path_links = datalocation + "links.csv"
file_path_ratings = datalocation + "ratings.csv"
file_path_tags = datalocation + "tags.csv"
file_path_genome_tags = datalocation + "genome-tags.csv"
file_path_genome_scores = datalocation + "genome-scores.csv"

### Load the data and review

Let's load each file in turn and observe, just to get a sense of familiarity with the data.  

#### Movies

In [15]:
movies_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_movies)
    .load(file_path_movies)
)

In [16]:
# Spark collects all transformations needed
# and execution doesn't begin until an "action" is triggered
# 
# 'show' triggers a partial execution 
#  'show' - limiting computation (where relevant) to the number of rows you want to display
movies_raw.show(10, False)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
|6      |Heat (1995)                       |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                    |Comedy|Romance                             |
|8      |Tom and Huck (1995)               |Adventure|Children                         |
|9      |Sudden Death

#### Links

In [17]:
links_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_links)
    .load(file_path_links)
)

In [18]:
links_raw.show(10, False)

+-------+-------+------+
|movieId|imdbId |tmdbId|
+-------+-------+------+
|1      |0114709|862   |
|2      |0113497|8844  |
|3      |0113228|15602 |
|4      |0114885|31357 |
|5      |0113041|11862 |
|6      |0113277|949   |
|7      |0114319|11860 |
|8      |0112302|45325 |
|9      |0114576|9091  |
|10     |0113189|710   |
+-------+-------+------+
only showing top 10 rows



#### Ratings

In [19]:
ratings_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_ratings)
    .load(file_path_ratings)
)

In [20]:
ratings_raw.show(10, False)

+------+-------+------+----------+
|userId|movieId|rating|timestamp |
+------+-------+------+----------+
|1     |296    |5.0   |1147880044|
|1     |306    |3.5   |1147868817|
|1     |307    |5.0   |1147868828|
|1     |665    |5.0   |1147878820|
|1     |899    |3.5   |1147868510|
|1     |1088   |4.0   |1147868495|
|1     |1175   |3.5   |1147868826|
|1     |1217   |3.5   |1147878326|
|1     |1237   |5.0   |1147868839|
|1     |1250   |4.0   |1147868414|
+------+-------+------+----------+
only showing top 10 rows



#### Tags

In [21]:
tags_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_tags)
    .load(file_path_tags)
)

In [22]:
tags_raw.show(10, False)

+------+-------+-----------------------+----------+
|userId|movieId|tag                    |timestamp |
+------+-------+-----------------------+----------+
|3     |260    |classic                |1439472355|
|3     |260    |sci-fi                 |1439472256|
|4     |1732   |dark comedy            |1573943598|
|4     |1732   |great dialogue         |1573943604|
|4     |7569   |so bad it's good       |1573943455|
|4     |44665  |unreliable narrators   |1573943619|
|4     |115569 |tense                  |1573943077|
|4     |115713 |artificial intelligence|1573942979|
|4     |115713 |philosophical          |1573943033|
|4     |115713 |tense                  |1573943042|
+------+-------+-----------------------+----------+
only showing top 10 rows



#### Tag Genome

In [23]:
genome_tags_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_genome_tags)
    .load(file_path_genome_tags)
)

In [24]:
genome_tags_raw.show(10, False)

+-----+------------+
|tagId|tag         |
+-----+------------+
|1    |007         |
|2    |007 (series)|
|3    |18th century|
|4    |1920s       |
|5    |1930s       |
|6    |1950s       |
|7    |1960s       |
|8    |1970s       |
|9    |1980s       |
|10   |19th century|
+-----+------------+
only showing top 10 rows



#### Tag Genome Scores

In [25]:
genome_scores_raw = (
    spark.read.format("csv")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("sep", ",")
    .option("escape", '"')
    .schema(schema_genome_scores)
    .load(file_path_genome_scores)
)

In [26]:
genome_scores_raw.show(10, False)

+-------+-----+---------+
|movieId|tagId|relevance|
+-------+-----+---------+
|1      |1    |0        |
|1      |2    |0        |
|1      |3    |0        |
|1      |4    |0        |
|1      |5    |0        |
|1      |6    |0        |
|1      |7    |0        |
|1      |8    |0        |
|1      |9    |0        |
|1      |10   |0        |
+-------+-----+---------+
only showing top 10 rows



# Harder questions on `MovieLens` using `PySpark`

## Question Set

For each question below, I provide a 'theme' in brackets - helps one think of the possible approaches and classify the 'kind' of query or analysis we are trying to crack.

### **1.  Transitive Genre Preference Propagation (Graph-like + Optimization)**    
    
---    
    
#### Question    
       
        
*   A user directly rates movies. We define *indirect* genre preference as follows: If User A rates 5 movies of Genre X highly (>=4 stars), and User A also rates 5 movies of Genre Y highly, then all users who rated at least 3 movies of Genre Y highly *also* indirectly prefer Genre X. Propagate this preference transitively.  Output the top 5 *indirectly* preferred genres for each user that are NOT directly preferred.    
---
    

### **2.  Time-Decayed Movie Recommendation Similarity (Windowing + Complex-ish Aggregation)**    
    
---    
    
#### Question    
       
        
*   Calculate a time-decayed similarity score between all pairs of movies.  The similarity is based on users who have rated both movies.  For each user who rated both, the contribution to similarity decays exponentially with the time difference between their ratings of the two movies (half-life of 30 days).  Output the top 10 most similar movies for each movie.    
---


### **3.  Genre Co-occurrence Network Analysis (Graph-like + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Construct a weighted, undirected graph where nodes are genres, and an edge exists between two genres if at least 100 users have rated at least 5 movies of *each* genre with an average rating of 4 or higher.  The edge weight is the Jaccard similarity of the user sets who contributed to each genre's qualification. Output the top 3 most "central" genres based on degree centrality (weighted).    
---
 

### **4.  Rating Pattern Anomaly Detection (Windowing + Custom Logic)**    
    
---    
    
#### Question    
       
        
*   Identify users whose rating behavior shows significant *sudden* shifts.  A "shift" is defined as a change in the average rating of at least 1.5 stars (up or down) over a rolling 7-day window, compared to the previous 30-day average, occurring in at least 3 distinct weeks within the dataset.    
---
   

### **5.  Movie Recommendation Cold Start (Complex Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   For each movie with *fewer* than 50 ratings, identify the top 5 "similar" movies based on a combination of shared genres (weighted by the number of shared genres) and the average rating difference of users who have rated both (penalize large differences).  Handle the case where no users have rated both.    
---
    

### **6.  Transitive User Similarity (Graph-like + Optimization)**    
    
---    
    
#### Question    
       
        
*   Calculate a transitive user similarity score.  If User A and User B have rated at least 5 movies in common with an average rating difference of less than 1, they have a direct similarity.  Transitive similarity extends this: If A is similar to B, and B is similar to C, then A and C have a transitive similarity score (decayed by a factor of 0.5 for each hop).  Output the 5 most *transitively* similar users for each user.    
---
    

### **7.  Time-Based Genre Evolution (Windowing + Complex Aggregation)**    
    
---    
    
#### Question    
       
        
*   Track the popularity of each genre over time (monthly windows).  Define "popularity" as the average rating of movies of that genre, weighted by the number of ratings in that month.  Identify genres that show a statistically significant (e.g., using a z-score) increase or decrease in popularity over any 6-month period.    
---
    
      

### **8.  User Rating Bias Detection (Aggregation + Statistical Analysis)**    
    
---    
    
#### Question    
       
        
*   Identify users who exhibit a significant rating bias (e.g., consistently rating higher or lower than the average).  Calculate each user's average rating deviation from the movie's average rating, and then identify outliers (e.g., users whose average deviation is more than 2 standard deviations from the mean deviation).    
---
    

### **9.  Movie Recommendation Diversification (Complex Filtering + Optimization)**    
    
---    
    
#### Question    
       
        
*   Given a user's top 10 recommended movies (based on a simple collaborative filtering model - assume this is pre-computed), re-rank these movies to increase genre diversity.  Penalize movies that share genres with already-recommended movies.    
---
    

### **10. Long-Tail Movie Recommendation (Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Identify "long-tail" movies (movies with few ratings but high average ratings).  For each user, recommend the top 3 long-tail movies that *do not* belong to genres the user has frequently rated.    
---
    

### **11.  Movie Release Year Influence (Windowing + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Analyze if the release year of a movie influences its ratings, considering user age.  Group users by age brackets (e.g., 18-25, 26-35, etc.) and movies by release year brackets. Calculate the average rating for each user-age/movie-year combination. Identify statistically significant correlations.    
---
    

### **12.  Genre Combination Preference (Aggregation + Complex Logic)**    
    
---    
    
#### Question    
       
        
*   Identify combinations of *two* genres that, when present together in a movie, result in significantly higher average ratings than movies with only one of those genres.    
---
    
     

### **13.  User Segmentation by Rating Patterns (Clustering - Conceptual, more like a discussion, *ignore this one in the first run*)**    

    I put the question here, as there's a separate module I am planning on clustering, MLLib and GraphX, this question whets your appetite.
---    
    
#### Question    
       
        
*   ***Describe* how you would use PySpark to segment users into distinct clusters based on their rating patterns (e.g., users who mostly give high ratings, users who are very critical, users with bimodal distributions, etc.).**
*   NOTE: You don't need to implement a full clustering algorithm, but outline the features you would extract and the PySpark operations you would use.   
---
    

### **14.  Movie Similarity via Rating Vector Cosine Similarity (Aggregation + Optimization)**    
    
---    
    
#### Question    
       
        
*   Calculate the cosine similarity between all pairs of movies based on their *user rating vectors*.  Handle cases where users have rated one movie but not the other (impute a default rating or use a smoothing technique).  Output the 10 most similar movies for each movie.    
---
    

### **15.  Impact of "Super-Raters" (Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Identify "super-raters" (users who have rated a significantly higher number of movies than average).  Analyze the impact of removing their ratings on the overall average rating of movies.  Are there specific genres or movies that are disproportionately affected?    
---
    
       

### **16.  Predictive Rating Decay (Windowing + Regression - Conceptual, ummm...)**    
    ...again just embedding thought experiments here, these would be too large or costly to build as a part of this assessment
    
---  
    
#### Question    
       
        
*   *Describe* how you would use PySpark to build a model that predicts how a movie's average rating will change over time (e.g., will it increase, decrease, or remain stable?). Consider factors like initial ratings, genre, and the ratings of similar movies.  Outline the features and PySpark operations.    
---
    

### **17.  Transitive Genre Dislike Propagation (Graph-like + Optimization, Negative Preference)**    
    
---    
    
#### Question    
       
        
*   Similar to Question 1, but propagate *dislike*.  If a user consistently rates movies of Genre X poorly (<=2 stars), and they also rate movies of Genre Y poorly, then users who dislike Genre Y also indirectly dislike Genre X.  Output the top 5 indirectly *disliked* genres for each user.    
---
    

### **18.  Genre Influence Over Time (Windowing + Correlation)**    
    
---    
    
#### Question    
       
        
*    Investigate the changing influence of genres on each other over time.  For each pair of genres, calculate the correlation between their average monthly ratings over a rolling 12-month window.  Identify genre pairs with increasingly strong positive or negative correlations.    
---
    
        

### **19.  User Rating Consistency Across Genres (Aggregation + Statistical Analysis)**    
    
---    
    
#### Question    
       
        
*   Analyze whether users maintain consistent rating behavior across different genres.  For each user and genre, calculate the standard deviation of their ratings.  Identify users whose rating consistency varies significantly across genres.    
---
    

### **20.  Movie Recommendation "Serendipity" (Complex Filtering + Optimization)**    
    
---    
    
#### Question    
       
        
*   Design a metric to measure the "serendipity" of a movie recommendation.  A serendipitous recommendation is one that is both relevant (high predicted rating) and *unexpected* (not belonging to the user's frequently rated genres or similar to movies they've already seen).  Implement this metric.    
---
    
        

### **21.  Temporal Rating Bias (Windowing + Statistical Analysis)**    
    
---    
    
#### Question    
       
        
*   Investigate if there's a temporal bias in user ratings (e.g., do users tend to rate higher or lower on weekends vs. weekdays, or during certain times of the year?).    
---
    

### **22.  Influence of Early Ratings (Windowing + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Analyze the influence of a movie's *early* ratings (e.g., the first 100 ratings) on its long-term average rating.  Do movies with high initial ratings tend to maintain high ratings, or is there a regression to the mean?    
---
    
        

### **23.  Genre Hybridity and Rating (Complex Filtering + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Define a measure of "genre hybridity" for a movie (e.g., the number of distinct genres it belongs to, or a more sophisticated measure based on genre co-occurrence).  Investigate the relationship between genre hybridity and average rating.    
---
    

### **24.  User "Criticality" Evolution (Windowing + Aggregation)**    
    
---    
    
#### Question    
       
        
*   Track how a user's "criticality" changes over time.  Define "criticality" as the difference between a user's average rating and the average rating of all users for the movies they've rated.  Analyze if users tend to become more or less critical over time.    
---
    
        

### **25.  Network Effects in Ratings (Graph-like + Optimization)**    
    
---    
    
#### Question    
       
        
*   Construct a user-movie bipartite graph.  Investigate if there are "network effects" in ratings – that is, if a user is connected to many users who have rated a movie highly, are they more likely to rate that movie highly as well, even after controlling for the movie's overall average rating?    
---
    

# Clear cache and stop the spark cluster

In [ ]:
# clear cache
spark.catalog.clearCache()

In [ ]:
# stop spark
spark.stop()